# Leren Excel 라벨 기반 정밀도 EDA 및 키워드 실험

이 노트북은 `라벨링.xlsx`를 기준으로 문제 생성용 자료와 AI 튜터 검색용 자료를 분리해 분석합니다.

- 문제 생성: `확정 골드 = 2`를 양성으로 사용하여 Precision을 우선 확인합니다.
- 튜터 검색: `확정 골드 = 1 또는 2`를 양성으로 사용하여 Recall을 함께 확인합니다.
- 키워드 매칭은 최종 판정이 아니라 후보 추출 단계입니다.
- 파일럿 셀은 동일 모델의 기존 결과를 재사용하고, 결과가 없을 때만 10건을 호출합니다.
- 독립 검증 전의 현재 골드는 참고 기준으로만 사용하며, LLM 결과만으로 Precision·Recall을 확정하지 않습니다.

In [1]:
from pathlib import Path
import json
import os
import sys

# 노트북을 저장소 루트·notebook 디렉터리 중 어디에서 실행해도 동작하게 합니다.
ROOT = Path.cwd()
while not (ROOT / 'app').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
# Settings가 저장소 루트의 .env를 읽도록 Notebook 작업 디렉터리를 고정합니다.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.rebuild_precision_pipeline import run

# 기존 재사용 모듈을 호출합니다. Excel 원본은 읽기만 하고 수정하지 않습니다.
summary = run()
print('입력 파일:', summary['input_file'])
print('전체 행 수:', summary['row_count'])
print('검수 큐:', summary['review_queue_count'], '건')

입력 파일: 라벨링.xlsx
전체 행 수: 160
검수 큐: 94 건


## 0. LLM 소규모 파일럿 실험

현재 EDA는 키워드 기준선을 측정하는 오프라인 단계입니다. 아래 파일럿은 키워드 후보와 비후보를 함께 최대 10건 선정하여 LLM의 판단 이유와 원문 근거를 확인합니다.

이 셀은 기존의 동일 모델 결과가 있으면 재사용하고, 결과가 없을 때만 소규모 OpenRouter 파일럿을 실행합니다. 다시 호출하려면 `FORCE_RERUN_LLM = True`로 바꿉니다. 결과는 Git에 올리지 않는 로컬 JSONL 파일에 저장되며, 이 결과만으로 최종 Precision·Recall을 주장하지 않습니다.

In [2]:
import importlib
import scripts.precision_llm_pilot as precision_llm_pilot
from app.core.config import get_settings

# 이미 열린 Jupyter 커널이 예전 모듈을 기억하고 있어도 최신 함수를 사용합니다.
precision_llm_pilot = importlib.reload(precision_llm_pilot)
get_settings.cache_clear()

# 실제 LLM 파일럿 결과를 반드시 표시합니다. 동일 결과가 있으면 재호출하지 않습니다.
RUN_LLM_EXPERIMENT = True
FORCE_RERUN_LLM = False
PILOT_SAMPLE_SIZE = 10

pilot_output = ROOT / 'data' / 'labeled' / 'precision' / 'llm_pilot_results.jsonl'
current_model = get_settings().llm_model
pilot_summary = None
if RUN_LLM_EXPERIMENT and not FORCE_RERUN_LLM:
    pilot_summary = precision_llm_pilot.load_existing_pilot(
        pilot_output, current_model
    )
if pilot_summary is None:
    pilot_summary = precision_llm_pilot.run(
        chapter_path=ROOT / 'configs' / 'chapters.yaml',
        output_path=pilot_output,
        sample_size=PILOT_SAMPLE_SIZE,
        execute=RUN_LLM_EXPERIMENT,
    )
print('요청 모델:', current_model)
print('Provider 응답 모델:', pilot_summary.get('provider_response_models') or '응답 메타데이터 없음')
print('파일럿 상태:', pilot_summary['status'])
print('예정 샘플 수:', pilot_summary['sample_size'])
print('LLM 호출 수:', pilot_summary['llm_calls'])


요청 모델: openai/gpt-5.6-sol
Provider 응답 모델: ['openai/gpt-5.6-sol']
파일럿 상태: REUSED_EXISTING_LLM_PILOT
예정 샘플 수: 10
LLM 호출 수: 10


## 1. 데이터 규모·토큰·라벨 분포


In [3]:
token_stats = summary['text_eda']['token_count']
char_stats = summary['text_eda']['character_count']
print(f"기관별 건수: {summary['source_distribution']}")
print(f"설명 문자 수: 평균 {char_stats['average']:.1f}, 중앙값 {char_stats['median']}, 최대 {char_stats['max']}")
print(f"설명 토큰 수: 평균 {token_stats['average']:.1f}, 중앙값 {token_stats['median']}, 총 {token_stats['total']}")
print(f"토크나이저: {token_stats['tokenizer']}")
print(f"확정 골드 분포: {summary['gold_distribution']}")
print(f"평가자 A·B 일치율: {summary['label_quality']['reviewer_agreement_rate']:.1%}")
print(f"평가자 불일치: {summary['label_quality']['reviewer_disagreement_count']}건")
print(f"자료 품질 검토필요: {summary['label_quality']['quality_review_count']}건")

기관별 건수: {'KDI': 160}
설명 문자 수: 평균 325.8, 중앙값 279.5, 최대 1000
설명 토큰 수: 평균 193.1, 중앙값 169.5, 총 30901
토크나이저: gpt-4o-mini
확정 골드 분포: {'0': 98, '1': 55, '2': 7}
평가자 A·B 일치율: 79.6%
평가자 불일치: 10건
자료 품질 검토필요: 7건


## 2. 기본 키워드와 확장 키워드 비교

기본 키워드는 핵심어만 사용하고, 확장 키워드는 약어·연관어·반대 개념까지 포함합니다. 두 전략을 문제 생성과 튜터 검색 목적별로 따로 평가합니다.

In [4]:
def show_metric(strategy, usage, label):
    item = summary['keyword_evaluation'][strategy][usage]
    metric = item['metrics']
    print(
        f"{label:12} | 후보 {item['candidate_count']:3}건 | "
        f"Precision {metric['precision']:.1%} | Recall {metric['recall']:.1%} | "
        f"F1 {metric['f1']:.1%} | 혼동행렬 {item['matrix']}"
    )

for strategy, label in [('basic_core', '기본 키워드'), ('expanded', '확장 키워드')]:
    show_metric(strategy, 'question_generation', label + ' / 문제 생성')
    show_metric(strategy, 'tutor_retrieval', label + ' / 튜터 검색')

print('주의: 위 수치는 독립 골드가 아닌 현재 Excel 골드 기준선입니다.')

기본 키워드 / 문제 생성 | 후보   4건 | Precision 100.0% | Recall 57.1% | F1 72.7% | 혼동행렬 {'tp': 4, 'fp': 0, 'fn': 3, 'tn': 153}
기본 키워드 / 튜터 검색 | 후보   4건 | Precision 100.0% | Recall 6.5% | F1 12.1% | 혼동행렬 {'tp': 4, 'fp': 0, 'fn': 58, 'tn': 98}
확장 키워드 / 문제 생성 | 후보   5건 | Precision 100.0% | Recall 71.4% | F1 83.3% | 혼동행렬 {'tp': 5, 'fp': 0, 'fn': 2, 'tn': 153}
확장 키워드 / 튜터 검색 | 후보   5건 | Precision 100.0% | Recall 8.1% | F1 14.9% | 혼동행렬 {'tp': 5, 'fp': 0, 'fn': 57, 'tn': 98}
주의: 위 수치는 독립 골드가 아닌 현재 Excel 골드 기준선입니다.


In [5]:
from IPython.display import Markdown, display

# 그래프 대신 표를 사용해 멘토님과 교수님이 실제 수치를 바로 확인할 수 있도록 합니다.
# 모든 값은 앞 단계에서 Excel과 키워드 평가 결과를 다시 계산한 summary에서 가져옵니다.
gold = summary['gold_distribution']
quality = summary['label_quality']
token_stats = summary['text_eda']['token_count']

def show_table(headers, rows):
    """Jupyter 기본 Markdown으로 결과 표를 출력합니다."""
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    lines.extend('| ' + ' | '.join(str(value) for value in row) + ' |' for row in rows)
    display(Markdown('\n'.join(lines)))

overview_table = [
    ['전체 문서', summary['row_count'], 'Excel에서 실제로 읽은 행 수'],
    ['평균 설명 토큰 수', round(token_stats['average'], 1), token_stats['tokenizer']],
    ['확정 골드 0점', gold.get('0', gold.get(0, 0)), '문제 생성용 음성 기준'],
    ['확정 골드 1점', gold.get('1', gold.get(1, 0)), '튜터 검색의 간접 관련 기준'],
    ['확정 골드 2점', gold.get('2', gold.get(2, 0)), '문제 생성용 양성 기준'],
    ['평가자 A·B 일치율', f"{quality['reviewer_agreement_rate']:.1%}", '독립 골드가 아닌 현재 라벨 기준'],
    ['LLM 파일럿 호출 수', pilot_summary['llm_calls'], '소규모 실험 결과'],
]
print('### 핵심 데이터 및 검수 지표')
show_table(['지표', '값', '해석'], overview_table)

strategy_rows = []
for strategy, strategy_label in [('basic_core', '기본 키워드'), ('expanded', '확장 키워드')]:
    for usage, usage_label in [('question_generation', '문제 생성'), ('tutor_retrieval', '튜터 검색')]:
        item = summary['keyword_evaluation'][strategy][usage]
        metric = item['metrics']
        strategy_rows.append([
            strategy_label, usage_label, item['candidate_count'],
            f"{metric['precision']:.1%}", f"{metric['recall']:.1%}",
            f"{metric['f1']:.1%}", str(item['matrix'])
        ])

print('### 키워드 전략 비교')
show_table(['필터 전략', '사용 목적', '후보 수', 'Precision', 'Recall', 'F1', '혼동행렬'], strategy_rows)

provider_models = pilot_summary.get('provider_response_models') or ['확인 불가']
pilot_table = [
    ['요청 모델', current_model],
    ['Provider 응답 모델', ', '.join(provider_models)],
    ['파일럿 상태', pilot_summary['status']],
    ['호출 수', pilot_summary['llm_calls']],
    ['결과 파일', str(pilot_output.relative_to(ROOT))],
]
print('### LLM 파일럿 실행 현황')
show_table(['항목', '값'], pilot_table)

print('### 의사결정 요약')
print('1. 문제 생성: 확장 키워드로 후보를 넓게 수집한 뒤 LLM 분류와 Human Review를 적용합니다.')
print('2. 튜터 검색: 키워드 필터만으로 제한하지 않고 임베딩 검색과 챕터 태그를 함께 사용합니다.')
print('3. 해석 주의: 현재 Precision·Recall은 독립적으로 확정된 골드가 아닌 Excel 라벨 기준선입니다.')

### 핵심 데이터 및 검수 지표


| 지표 | 값 | 해석 |
| --- | --- | --- |
| 전체 문서 | 160 | Excel에서 실제로 읽은 행 수 |
| 평균 설명 토큰 수 | 193.1 | gpt-4o-mini |
| 확정 골드 0점 | 98 | 문제 생성용 음성 기준 |
| 확정 골드 1점 | 55 | 튜터 검색의 간접 관련 기준 |
| 확정 골드 2점 | 7 | 문제 생성용 양성 기준 |
| 평가자 A·B 일치율 | 79.6% | 독립 골드가 아닌 현재 라벨 기준 |
| LLM 파일럿 호출 수 | 10 | 소규모 실험 결과 |

### 키워드 전략 비교


| 필터 전략 | 사용 목적 | 후보 수 | Precision | Recall | F1 | 혼동행렬 |
| --- | --- | --- | --- | --- | --- | --- |
| 기본 키워드 | 문제 생성 | 4 | 100.0% | 57.1% | 72.7% | {'tp': 4, 'fp': 0, 'fn': 3, 'tn': 153} |
| 기본 키워드 | 튜터 검색 | 4 | 100.0% | 6.5% | 12.1% | {'tp': 4, 'fp': 0, 'fn': 58, 'tn': 98} |
| 확장 키워드 | 문제 생성 | 5 | 100.0% | 71.4% | 83.3% | {'tp': 5, 'fp': 0, 'fn': 2, 'tn': 153} |
| 확장 키워드 | 튜터 검색 | 5 | 100.0% | 8.1% | 14.9% | {'tp': 5, 'fp': 0, 'fn': 57, 'tn': 98} |

### LLM 파일럿 실행 현황


| 항목 | 값 |
| --- | --- |
| 요청 모델 | openai/gpt-5.6-sol |
| Provider 응답 모델 | openai/gpt-5.6-sol |
| 파일럿 상태 | REUSED_EXISTING_LLM_PILOT |
| 호출 수 | 10 |
| 결과 파일 | data/labeled/precision/llm_pilot_results.jsonl |

### 의사결정 요약
1. 문제 생성: 확장 키워드로 후보를 넓게 수집한 뒤 LLM 분류와 Human Review를 적용합니다.
2. 튜터 검색: 키워드 필터만으로 제한하지 않고 임베딩 검색과 챕터 태그를 함께 사용합니다.
3. 해석 주의: 현재 Precision·Recall은 독립적으로 확정된 골드가 아닌 Excel 라벨 기준선입니다.


## 3. 실험 해석 및 다음 단계

현재 결과에서는 확장 키워드가 문제 생성 후보의 재현율을 높였지만, 튜터 검색 목적에는 여전히 재현율이 낮습니다. 따라서 문제 생성은 키워드 후보 뒤에 LLM 정밀 분류와 Human Review를 두고, 튜터 검색은 키워드로 과도하게 제한하지 않고 임베딩 검색과 챕터 태그를 함께 사용하는 방향이 적절합니다.

평가자 A 라벨이 확정 골드와 동일하므로 현재 수치는 모델의 독립 성능이 아닙니다. 다음 실험에서는 평가자 A·B 불일치 10건, 자료 품질 검토필요 7건, 직접 관련 양성 표본을 포함한 독립 골드 세트를 확정한 뒤 같은 Notebook에서 임계값별 Precision·Recall 변화를 비교해야 합니다.